# Notebook 2: Sentinel-1/2/3 GEE Exports to NPZ


In [ ]:
import os, glob, gc, re, json, warnings
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from tqdm import tqdm

import rasterio
warnings.filterwarnings('ignore')

GEE_DIR       = r"./data/GEE_exports"              # PAIR_*.tif from GEE
OUTPUT_DIR    = "./processed_sentinel_npz"    # where .npz files go
FIGURES_DIR   = "./figures_sentinel"

for d in [OUTPUT_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"GEE dir:    {GEE_DIR}")
print(f"Output dir: {OUTPUT_DIR}")


## 1) Catalog GEE Pair GeoTIFFs

In [ ]:
def parse_pair_filename(filepath):
    m = re.match(r'PAIR_(\d{8})_(\d{8})', Path(filepath).stem)
    if m:
        return datetime.strptime(m.group(1), '%Y%m%d'), datetime.strptime(m.group(2), '%Y%m%d')
    return None, None

def catalog_gee(gee_dir):
    tifs = sorted(glob.glob(os.path.join(gee_dir, "PAIR_*.tif")))
    catalog = []
    for fp in tifs:
        t1, t2 = parse_pair_filename(fp)
        if t1:
            catalog.append({
                'filepath': fp, 'filename': Path(fp).name,
                't1': t1, 't2': t2,
                'dt_days': (t2 - t1).days,
                't1_str': t1.strftime('%Y-%m-%d'),
                't2_str': t2.strftime('%Y-%m-%d'),
                'size_mb': os.path.getsize(fp) / 1e6,
            })
    catalog.sort(key=lambda x: x['t1'])
    
    print(f"Found {len(catalog)} pair GeoTIFFs")
    if catalog:
        total_gb = sum(f['size_mb'] for f in catalog) / 1000
        print(f"  Date range: {catalog[0]['t1_str']} -> {catalog[-1]['t2_str']}")
        print(f"  Total size: {total_gb:.2f} GB")
        print(f"  Pair gaps: {sorted(set(f['dt_days'] for f in catalog))} days")
    return catalog

pair_catalog = catalog_gee(GEE_DIR)


## 2) Inspect Band Structure


In [ ]:
def inspect_pair_geotiff(filepath, max_bands_print=40):
    with rasterio.open(filepath) as src:
        print(f"File: {Path(filepath).name}")
        print(f"  CRS: {src.crs}")
        print(f"  Resolution: {src.res}")
        print(f"  Size: {src.width} x {src.height} ({src.width * src.height / 1e6:.1f}M pixels)")
        print(f"  Bands: {src.count}")
        print(f"  Dtype: {src.dtypes[0]}")
        
        band_info = []
        for i in range(src.count):
            desc = src.descriptions[i] or f'band_{i+1}'
            band_info.append(desc)
        
        t1_bands = [b for b in band_info if b.endswith('_t1')]
        t2_bands = [b for b in band_info if b.endswith('_t2')]
        other = [b for b in band_info if not b.endswith('_t1') and not b.endswith('_t2')]
        
        print(f"\n  _t1 bands ({len(t1_bands)}): {t1_bands[:20]}")
        print(f"  _t2 bands ({len(t2_bands)}): {t2_bands[:20]}")
        if other: print(f"  Other bands ({len(other)}): {other[:10]}")
        
        print(f"\n  Band data ranges (sampling {min(src.count, 10)} bands):")
        for i in range(0, min(src.count, 10)):
            data = src.read(i + 1)
            desc = src.descriptions[i] or f'band_{i+1}'
            nz = np.count_nonzero(data) / data.size * 100
            print(f"    {desc:35s}  [{data.min():12.4f}, {data.max():12.4f}]  {nz:5.1f}% nonzero")
        
        return band_info

if pair_catalog:
    all_bands = inspect_pair_geotiff(pair_catalog[0]['filepath'])


## 3) Define Consistent Channel Ordering


In [ ]:
INPUT_DATA_BANDS = [
    'S1_CO_asc', 'S1_CX_asc', 'angle_asc',
    'S1_CO_desc', 'S1_CX_desc', 'angle_desc',
    'B2', 'B3', 'B4', 'B8', 'B11', 'B12',
    'Oa04_radiance', 'Oa06_radiance', 'Oa08_radiance',
    'doy',
]

MASK_BANDS = ['s1_valid_asc', 's1_valid_desc', 's2_valid', 's3_valid']

FLAG_BANDS = [
    's1_used_vv_asc', 's1_used_hh_asc',
    's1_used_vv_desc', 's1_used_hh_desc',
    's1_is_asc_asc', 's1_is_desc_desc',
]

ALL_BANDS = INPUT_DATA_BANDS + MASK_BANDS + FLAG_BANDS
N_CHANNELS = len(ALL_BANDS)

print(f"Channel structure: {N_CHANNELS} channels per timestep")
for i, bn in enumerate(ALL_BANDS):
    cat = 'DATA' if bn in INPUT_DATA_BANDS else 'MASK' if bn in MASK_BANDS else 'FLAG'
    print(f"  ch {i:2d}: {bn:30s} [{cat}]")


## 4) Spatial & Temporal Sensor Coverage


In [ ]:
def analyze_sensor_coverage(pair_catalog, max_pairs=None):
    if not pair_catalog: 
        print("No pairs to analyze"); return
    
    pairs = pair_catalog[:max_pairs] if max_pairs else pair_catalog
    
    stats = []
    
    for pi, pair in enumerate(tqdm(pairs, desc="Scanning sensor coverage")):
        try:
            with rasterio.open(pair['filepath']) as src:
                descs = [src.descriptions[i] for i in range(src.count)]
                name_to_idx = {d: i+1 for i, d in enumerate(descs) if d}
                
                row = {'date': pair['t1_str'], 'month': pair['t1'].month}
                
                for sensor, mask_band in [('S1_ASC', 's1_valid_asc_t1'),
                                           ('S1_DESC', 's1_valid_desc_t1'),
                                           ('S2', 's2_valid_t1'),
                                           ('S3', 's3_valid_t1')]:
                    if mask_band in name_to_idx:
                        mask = src.read(name_to_idx[mask_band])
                        frac = (mask > 0.5).sum() / mask.size * 100
                    else:
                        frac = 0.0
                    row[sensor] = frac
                
                stats.append(row)
        except Exception as e:
            print(f"Skipping corrupt or unreadable file {pair['filepath']}: {e}")
            continue
    
    if not stats:
        print("No valid stats to plot.")
        return []
        
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    import numpy as np
    from collections import defaultdict
    import os
    
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    
    dates = [datetime.strptime(s['date'], '%Y-%m-%d') for s in stats]
    sensors = ['S1_ASC', 'S1_DESC', 'S2', 'S3']
    colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#d62728']
    
    for sensor, color in zip(sensors, colors):
        vals = [s.get(sensor, 0) for s in stats]
        axes[0].plot(dates, vals, 'o-', label=sensor, color=color, markersize=4, alpha=0.8)
    
    axes[0].set_ylabel('Valid coverage (%)')
    axes[0].set_title('Sensor Coverage Over Time (t1)')
    axes[0].legend()
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    axes[0].set_ylim(-5, 105)
    
    by_month = defaultdict(lambda: defaultdict(list))
    for s in stats:
        for sensor in sensors:
            by_month[s['month']][sensor].append(s.get(sensor, 0))
    
    months = sorted(by_month.keys())
    x = np.arange(len(months))
    width = 0.2
    for i, (sensor, color) in enumerate(zip(sensors, colors)):
        means = [np.mean(by_month[m][sensor]) for m in months]
        axes[1].bar(x + i * width, means, width, label=sensor, color=color, alpha=0.8)
    
    axes[1].set_xticks(x + width * 1.5)
    axes[1].set_xticklabels([f'{m:02d}' for m in months])
    axes[1].set_xlabel('Month')
    axes[1].set_ylabel('Avg Valid Coverage (%)')
    axes[1].set_title('Monthly Sensor Availability')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'sensor_coverage.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    return stats


In [ ]:
def analyze_temporal_completeness(catalog):
    if not catalog: return
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    import numpy as np
    from collections import defaultdict
    import os
    
    dates_start = sorted([v['t1'] for v in catalog])
    print(f"Total pairs: {len(catalog)}")
    print(f"Date range: {dates_start[0].strftime('%Y-%m-%d')} → {dates_start[-1].strftime('%Y-%m-%d')}")
    
    if len(dates_start) > 1:
        gaps = [(dates_start[i+1] - dates_start[i]).days for i in range(len(dates_start)-1)]
        print(f"Gap stats: min={min(gaps)}d, max={max(gaps)}d, median={np.median(gaps):.0f}d")        
        
    by_year = defaultdict(int)
    for d in dates_start: by_year[d.year] += 1
    print(f"\nPairs per year:")
    for yr in sorted(by_year.keys()): print(f"  {yr}: {by_year[yr]}")
        
    by_month = defaultdict(int)
    for d in dates_start: by_month[d.strftime('%Y-%m')] += 1
        
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    axes[0].scatter(dates_start, [1]*len(dates_start), marker='|', s=200, color='steelblue')
    axes[0].set_title('Sentinel Pair Timeline (t1)')
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    axes[0].set_yticks([])
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45)
    
    months = sorted(by_month.keys())
    counts = [by_month[m] for m in months]
    axes[1].bar(range(len(months)), counts, color='steelblue')
    step = max(1, len(months)//20)
    axes[1].set_xticks(range(0, len(months), step))
    axes[1].set_xticklabels([months[i] for i in range(0, len(months), step)], rotation=45)
    axes[1].set_title('Pairs per Month'); axes[1].set_ylabel('Count')
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'temporal_completeness.png'), dpi=150, bbox_inches='tight')     
    plt.show()

analyze_temporal_completeness(pair_catalog)


In [ ]:
def analyze_spatial_coverage_sample(pair_info):
    import rasterio
    import matplotlib.pyplot as plt
    import numpy as np
    import os
    
    print(f"Analyzing spatial coverage for {pair_info['filename']}")
    try:
        with rasterio.open(pair_info['filepath']) as src:
            descs = [src.descriptions[i] for i in range(src.count)]
            name_to_idx = {d: i+1 for i, d in enumerate(descs) if d}
            
            def read_band(name):
                if name in name_to_idx:
                    return src.read(name_to_idx[name])
                return np.zeros((src.height, src.width))
            
            s1_asc = read_band('S1_CO_asc_t1')
            s1_desc = read_band('S1_CO_desc_t1')
            s1_v_asc = read_band('s1_valid_asc_t1')
            s2_v = read_band('s2_valid_t1')
            s2_b3 = read_band('B3_t1') # green band
            
            s1_v_asc = s1_v_asc > 0
            s2_v = s2_v > 0
            
            fig, axes = plt.subplots(1, 3, figsize=(18, 6))
            axes[0].imshow(s1_asc, cmap='gray', vmin=-25, vmax=0)
            axes[0].set_title('S1 ASC CO')
            
            axes[1].imshow(s1_v_asc.astype(float), cmap='RdYlGn', vmin=0, vmax=1)
            axes[1].set_title(f'S1 ASC valid ({s1_v_asc.sum() / s1_v_asc.size * 100:.1f}%)')
            
            axes[2].imshow(s2_v.astype(float), cmap='RdYlGn', vmin=0, vmax=1)
            axes[2].set_title(f'S2 valid ({s2_v.sum() / s2_v.size * 100:.1f}%)')
            
            plt.tight_layout()
            plt.savefig(os.path.join(FIGURES_DIR, 'spatial_coverage_sample.png'), dpi=150, bbox_inches='tight')
            plt.show()
    except Exception as e:
        print(f"Could not read {pair_info['filename']}: {e}")

if pair_catalog:
    for p in pair_catalog:
        if p['t1'].month in [6, 7, 8]:
            analyze_spatial_coverage_sample(p)
            break


## 5) Convert GeoTIFFs to `.npz`


In [ ]:
def geotiff_to_npz(pair_info, band_list=ALL_BANDS, output_dir=OUTPUT_DIR):
    with rasterio.open(pair_info['filepath']) as src:
        descs = [src.descriptions[i] for i in range(src.count)]
        name_to_idx = {d: i+1 for i, d in enumerate(descs) if d}
        
        H, W = src.height, src.width
        C = len(band_list)
        x1 = np.zeros((H, W, C), dtype=np.float32)
        x2 = np.zeros((H, W, C), dtype=np.float32)
        
        for ch, band in enumerate(band_list):
            for sfx, arr in [('_t1', x1), ('_t2', x2)]:
                key = f"{band}{sfx}"
                if key in name_to_idx:
                    data = src.read(name_to_idx[key]).astype(np.float32)
                    arr[:, :, ch] = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)
        
        meta = {
            'crs': str(src.crs),
            'transform': list(src.transform)[:6],
            'shape': [H, W],
            'channels': band_list,
            'n_channels': C,
            't1': pair_info['t1_str'],
            't2': pair_info['t2_str'],
            'dt_days': pair_info['dt_days'],
            'bounds': list(src.bounds),
        }
    
    out_path = os.path.join(output_dir, f"{Path(pair_info['filepath']).stem}.npz")
    np.savez_compressed(out_path, x1=x1, x2=x2, metadata=json.dumps(meta))
    
    size_mb = os.path.getsize(out_path) / 1e6
    return out_path, size_mb


def convert_all_sentinel(catalog, output_dir=OUTPUT_DIR):
    print(f"Converting {len(catalog)} GeoTIFFs to .npz...")
    paths = []
    total_mb = 0
    
    for pair in tqdm(catalog, desc="Converting"):
        try:
            path, sz = geotiff_to_npz(pair, output_dir=output_dir)
            paths.append(path)
            total_mb += sz
        except Exception as e:
            print(f"  ERROR {pair['filename']}: {e}")
        gc.collect()
    
    print(f"\nConverted {len(paths)} / {len(catalog)} files")
    print(f"Total .npz size: {total_mb:.0f} MB ({total_mb/1000:.2f} GB)")
    return paths

sentinel_npz_paths = convert_all_sentinel(pair_catalog)


## 6) Verify `.npz` Output

In [ ]:
def verify_sentinel_npz(npz_path):
    data = np.load(npz_path, allow_pickle=True)
    meta = json.loads(str(data['metadata']))
    x1, x2 = data['x1'], data['x2']
    
    print(f"\n{Path(npz_path).name}")
    print(f"  x1: {x1.shape}, x2: {x2.shape}")
    print(f"  Dates: {meta['t1']} -> {meta['t2']} ({meta['dt_days']}d)")
    print(f"  CRS: {meta['crs']}, Shape: {meta['shape']}")
    
    channels = meta['channels']
    for i, ch in enumerate(channels):
        v1 = x1[:,:,i]; v2 = x2[:,:,i]
        nz1 = np.count_nonzero(v1) / v1.size * 100
        nz2 = np.count_nonzero(v2) / v2.size * 100
        print(f"  ch {i:2d} {ch:30s}  t1: {nz1:5.1f}% nz [{v1.min():.3f},{v1.max():.3f}]  "
              f"t2: {nz2:5.1f}% nz [{v2.min():.3f},{v2.max():.3f}]")
    data.close()

npz_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "PAIR_*.npz")))
if npz_files: verify_sentinel_npz(npz_files[0])


## Summary
